In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from math import sqrt

In [ ]:
def sklearn_metrick_printer(y_true, y_pred):
    print(f'MAE: {mean_absolute_error(y_true, y_pred)}')
    print(f'MSE: {mean_squared_error(y_true, y_pred)}')
    print(f'RMSE: {sqrt(mean_squared_error(y_true, y_pred))}')
    print(f'MAPE: {mean_absolute_percentage_error(y_true, y_pred)}')
    print(f'R^2: {r2_score(y_true, y_pred)}')

In [ ]:
data = pd.read_csv('data/r5_eda.csv')

In [ ]:
y = data['quality']
X = data.drop('quality', axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### DecisionTreeRegressor

In [ ]:
from sklearn.tree import DecisionTreeRegressor

reg = DecisionTreeRegressor(max_depth=6)
reg.fit(X_train, y_train)

In [ ]:
y_pred = reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score


def objective(trial):
    max_depth = trial.suggest_int('max_depth', 1, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])
    
    model = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
reg = DecisionTreeRegressor(**study.best_params, random_state=42)
reg.fit(X_train, y_train)

In [ ]:
y_pred = reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### BaggingRegressor

In [ ]:
from sklearn.ensemble import BaggingRegressor


base_regressor = DecisionTreeRegressor(max_depth=6)

bg_reg = BaggingRegressor(
    estimator=base_regressor,
    n_estimators=100,
    random_state=42
)

bg_reg.fit(X_train, y_train)

In [ ]:
y_pred = bg_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_samples = trial.suggest_float('max_samples', 0.1, 1.0)
    max_features = trial.suggest_float('max_features', 0.1, 1.0)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])
    
    model = BaggingRegressor(
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=42
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
bg_reg = BaggingRegressor(**study.best_params, random_state=42)
bg_reg.fit(X_train, y_train)

In [ ]:
y_pred = bg_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### GradientBoostingRegressor

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor


gb_reg = GradientBoostingRegressor(max_depth=6, n_estimators=100, random_state=42)
gb_reg.fit(X_train, y_train)

In [ ]:
y_pred = gb_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    max_depth = trial.suggest_int('max_depth', 1, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    
    model = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        subsample=subsample,
        random_state=42
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
gb_reg = GradientBoostingRegressor(**study.best_params, random_state=42)
gb_reg.fit(X_train, y_train)

In [ ]:
y_pred = gb_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### StackingRegressor

In [ ]:
from sklearn.linear_model import LinearRegression


estimators = [
    ('dt', DecisionTreeRegressor(max_depth=3)),
    ('lr', LinearRegression())
]

stack_reg = StackingRegressor(estimators=estimators, final_estimator=LinearRegression())
stack_reg.fit(X, y)

In [ ]:
y_pred = stack_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
from sklearn.linear_model import Ridge


def objective(trial):
    dt_max_depth = trial.suggest_int('dt_max_depth', 1, 10)
    dt_min_samples_split = trial.suggest_int('dt_min_samples_split', 2, 20)
    alpha = trial.suggest_float('alpha', 0.1, 10.0)
    
    estimators = [
        ('dt', DecisionTreeRegressor(max_depth=dt_max_depth, min_samples_split=dt_min_samples_split, random_state=42)),
        ('lr', LinearRegression())
    ]
    meta_model = Ridge(alpha=alpha, random_state=42)
    
    model = StackingRegressor(estimators=estimators, final_estimator=meta_model)
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
best_params = study.best_params
best_estimators = [
    ('dt', DecisionTreeRegressor(
        max_depth=best_params['dt_max_depth'], min_samples_split=best_params['dt_min_samples_split'], random_state=42)),
    ('lr', LinearRegression())
]
best_meta_model = Ridge(alpha=best_params['alpha'], random_state=42)
stack_reg = StackingRegressor(
    estimators=best_estimators, final_estimator=best_meta_model)
stack_reg.fit(X_train, y_train)

In [ ]:
y_pred = stack_reg.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### CatBoostRegressor

In [ ]:
from catboost import CatBoostRegressor


cat_model = CatBoostRegressor(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train, y_train)

In [ ]:
y_pred = cat_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    iterations = trial.suggest_int('iterations', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
    depth = trial.suggest_int('depth', 1, 10)
    l2_leaf_reg = trial.suggest_float('l2_leaf_reg', 1, 10)
    bagging_temperature = trial.suggest_float('bagging_temperature', 0, 1)
    
    model = CatBoostRegressor(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        bagging_temperature=bagging_temperature,
        random_seed=42,
        verbose=0
    )
    
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

In [ ]:
cat_model = CatBoostRegressor(**study.best_params, random_seed=42, verbose=0)
cat_model.fit(X_train, y_train)

In [ ]:
y_pred = cat_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### XGBRegressor

In [ ]:
from xgboost import XGBRegressor


xgb_model = XGBRegressor(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    # Определение гиперпараметров для поиска
    n_estimators = trial.suggest_int('n_estimators', 50, 300)  # Количество деревьев
    max_depth = trial.suggest_int('max_depth', 1, 10)  # Максимальная глубина деревьев
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)  # Скорость обучения
    subsample = trial.suggest_float('subsample', 0.5, 1.0)  # Доля выборки для обучения каждого дерева
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)  # Доля признаков для каждого дерева
    gamma = trial.suggest_float('gamma', 0, 5)  # Минимальное снижение потерь для разделения
    reg_alpha = trial.suggest_float('reg_alpha', 0, 10)  # L1-регуляризация
    reg_lambda = trial.suggest_float('reg_lambda', 0, 10)  # L2-регуляризация
    
    # Создание модели с предложенными гиперпараметрами
    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=42,  # Фиксированный random_state для воспроизводимости
        verbosity=0  # Без вывода сообщений
    )
    
    # Выполнение 5-кратной кросс-валидации с метрикой neg_mean_squared_error
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()  # Минимизация MSE (возвращаем отрицательное значение)

# Создание и оптимизация исследования Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # 50 итераций поиска

In [ ]:
xgb_model = XGBRegressor(**study.best_params, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred = xgb_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

### LGBMRegressor

In [ ]:
from lightgbm import LGBMRegressor


lgb_model = LGBMRegressor(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_model.fit(X_train, y_train)

In [ ]:
y_pred = lgb_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

#### Подбор гиперпараметров

In [ ]:
def objective(trial):
    # Определение гиперпараметров для поиска
    n_estimators = trial.suggest_int('n_estimators', 50, 300)  # Количество деревьев
    max_depth = trial.suggest_int('max_depth', 1, 10)  # Максимальная глубина деревьев
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)  # Скорость обучения
    num_leaves = trial.suggest_int('num_leaves', 2, 100)  # Максимальное число листьев в дереве
    min_child_samples = trial.suggest_int('min_child_samples', 1, 20)  # Минимальное число образцов в листе
    subsample = trial.suggest_float('subsample', 0.5, 1.0)  # Доля выборки для обучения каждого дерева
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)  # Доля признаков для каждого дерева
    reg_alpha = trial.suggest_float('reg_alpha', 0, 10)  # L1-регуляризация
    reg_lambda = trial.suggest_float('reg_lambda', 0, 10)  # L2-регуляризация
    
    # Создание модели с предложенными гиперпараметрами
    model = LGBMRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        seed=42,  # Фиксированный seed для воспроизводимости
        verbosity=-1  # Без вывода сообщений
    )
    
    # Выполнение 5-кратной кросс-валидации с метрикой neg_mean_squared_error
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    return -scores.mean()  # Минимизация MSE (возвращаем отрицательное значение)

# Создание и оптимизация исследования Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)  # 50 итераций поиска

In [ ]:
lgb_model = LGBMRegressor(**study.best_params, seed=42, verbosity=-1)
lgb_model.fit(X_train, y_train)

In [ ]:
y_pred = lgb_model.predict(X_test)

sklearn_metrick_printer(y_test, y_pred)

Все модели

In [ ]:
models = [reg, bg_reg, gb_reg, stack_reg, cat_model, xgb_model, lgb_model]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

# Предполагается, что X_test и y_test определены
model_names = [
    'Decision Tree Regressor',
    'Bagging Regressor',
    'Gradient Boosting Regressor',
    'Stacking Regressor',
    'CatBoost Regressor',
    'XGBoost Regressor',
    'LightGBM Regressor'
]

# Список для хранения результатов
results = []
for model, name in zip(models, model_names):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'MSE': mse, 'RMSE': rmse, 'R2': r2})

# Создание таблицы сравнения
df = pd.DataFrame(results)
df = df.sort_values(by='RMSE')  # Сортировка по RMSE (меньше — лучше)

# Вывод таблицы
print("Таблица сравнения:")
print(df)

# Построение графика
plt.figure(figsize=(12, 6))
plt.bar(df['Model'], df['RMSE'], color='skyblue')
plt.xlabel('Модель')
plt.ylabel('RMSE')
plt.title('Сравнение регрессионных моделей по RMSE')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(df['RMSE']):
    plt.text(i, v + 0.01 * max(df['RMSE']), f'{v:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn import tree

In [ ]:
fig = plt.figure(figsize=(15,10))
DT_plot = tree.plot_tree(reg, feature_names=X_train.columns, filled=True)